# 機器學習概念

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明機器學習如何透過資料學習規律，而不是完全依賴人工撰寫規則。
2. 區分監督式學習、非監督式學習與強化學習的資料條件與應用情境。
3. 使用 Python 建立簡單的分類、聚類與降維示範。
4. 觀察資料品質、特徵選擇與模型評估對結果的影響。
5. 透過 TODO 練習完成一個簡單的監督式學習流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節會用到的 Python 套件，並設定隨機種子，讓每次執行結果更容易重現。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score

np.random.seed(42)

print('環境設定完成')
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)


## 核心概念說明

機器學習是讓電腦從資料中找出規律，並將學到的規律用於預測、分類、分群或決策。和傳統程式設計不同，機器學習通常不是由人直接寫出所有判斷規則，而是讓模型根據資料自動調整參數。

### 1. 監督式學習

監督式學習使用「有標記資料」。每筆資料都有輸入特徵與正確答案，例如鳶尾花的花萼長度、花瓣寬度，以及它實際屬於哪一種花。常見任務包含：

- 分類：預測類別，例如是否違約、是否罹病、圖片是哪一類。
- 迴歸：預測連續數值，例如房價、銷售量、溫度。

### 2. 非監督式學習

非監督式學習使用「沒有標記資料」。模型需要自行找出資料內部結構，例如把顧客依消費行為分群，或把高維資料壓縮成較容易觀察的低維表示。

### 3. 強化學習

強化學習關注代理人在環境中的序列決策。代理人採取行動後會得到獎勵或懲罰，目標是透過試錯學習出能最大化長期累積獎勵的策略。


In [ ]:
# ── 示範：監督式學習分類 ──────────────────────────────
# 使用 sklearn 內建鳶尾花資料集，訓練一個 Logistic Regression 分類模型，示範「有標記資料」如何用於預測類別。

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=200)
model.fit(X_train_scaled, y_train)

pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)

print('分類任務：預測鳶尾花品種')
print('測試資料準確率:', round(accuracy, 3))
print('\n混淆矩陣:')
print(cm)

sample = pd.DataFrame(X_test[:5], columns=iris.feature_names)
sample['真實類別'] = [iris.target_names[i] for i in y_test[:5]]
sample['預測類別'] = [iris.target_names[i] for i in pred[:5]]
print('\n前 5 筆預測結果:')
print(sample)


## 為什麼不能只看模型是否能執行？

機器學習模型能成功訓練，不代表它真的可靠。實務上需要注意以下問題：

- 資料是否足夠代表真實情境。
- 訓練資料與測試資料是否適當分開。
- 特徵是否經過合理處理，例如尺度差異過大時可使用標準化。
- 評估指標是否符合任務目的，例如分類可看準確率、混淆矩陣，聚類可看輪廓係數。
- 模型是否具有可解釋性，尤其在醫療、金融與法律等高風險場域。

接下來的程式會示範非監督式學習如何在沒有標記答案的情況下發現資料結構。


In [ ]:
# ── 示範：非監督式學習聚類與降維 ──────────────────────────
# 建立沒有標記的二維資料，使用 K-Means 找出群組，再用 PCA 示範如何將較高維資料壓縮成二維以便觀察。

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

X_blob, _ = make_blobs(
    n_samples=180,
    centers=3,
    cluster_std=1.1,
    random_state=42
)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_blob)
score = silhouette_score(X_blob, cluster_labels)

plt.figure(figsize=(6, 4))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=cluster_labels, cmap='viridis', s=40)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    c='red', marker='X', s=160, label='群中心'
)
plt.title('K-Means 聚類示範')
plt.xlabel('特徵 1')
plt.ylabel('特徵 2')
plt.legend()
plt.show()

iris = load_iris()
X_scaled = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('K-Means 輪廓係數:', round(score, 3))
print('PCA 前兩個主成分保留的資訊比例:', np.round(pca.explained_variance_ratio_, 3))
print('PCA 前兩個主成分合計:', round(pca.explained_variance_ratio_.sum(), 3))


## 強化學習的簡化理解

強化學習包含三個核心要素：

- 代理：負責選擇行動的學習者。
- 環境：代理互動的對象，會回傳狀態與獎勵。
- 獎勵：用來表示行動結果好壞的回饋。

在真實應用中，強化學習可能用於遊戲 AI、機器人控制或自動化決策。本 Notebook 使用「多臂拉霸」作為輕量示範：代理每次選擇一個按鈕，每個按鈕有不同的平均獎勵，代理需要在探索新選項與利用已知好選項之間取得平衡。


In [ ]:
# ── 示範：強化學習的探索與利用 ───────────────────────────
# 使用簡化的 epsilon-greedy 策略模擬強化學習。代理有時探索不同選項，有時選擇目前估計獎勵最高的行動。

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

true_rewards = np.array([0.2, 0.5, 0.8])
estimated_rewards = np.zeros(3)
action_counts = np.zeros(3)
reward_history = []

epsilon = 0.1
rounds = 300

for step in range(rounds):
    if np.random.rand() < epsilon:
        action = np.random.randint(3)
    else:
        action = np.argmax(estimated_rewards)

    reward = np.random.normal(loc=true_rewards[action], scale=0.1)
    action_counts[action] += 1
    estimated_rewards[action] += (reward - estimated_rewards[action]) / action_counts[action]
    reward_history.append(reward)

print('真實平均獎勵:', true_rewards)
print('模型估計獎勵:', np.round(estimated_rewards, 3))
print('各行動被選擇次數:', action_counts.astype(int))
print('最常被選擇的行動:', int(np.argmax(action_counts)))

moving_average = np.convolve(reward_history, np.ones(20) / 20, mode='valid')
plt.figure(figsize=(6, 4))
plt.plot(moving_average)
plt.title('平均獎勵變化')
plt.xlabel('回合')
plt.ylabel('近 20 回合平均獎勵')
plt.show()


In [ ]:
# ── 自我測驗：完成監督式學習流程 ──────────────────────────
# 請依照 TODO 提示補齊程式，完成資料切分、標準化、模型訓練與準確率評估。

import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

# TODO 1: 使用 train_test_split 將資料分成訓練集與測試集。
# 條件：test_size=0.2, random_state=42, stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TODO 2: 建立 StandardScaler，並只用訓練資料 fit，再轉換訓練與測試資料。
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TODO 3: 建立 LogisticRegression(max_iter=200)，並用訓練資料訓練模型。
model = LogisticRegression(max_iter=200)
model.fit(X_train_scaled, y_train)

# TODO 4: 使用測試資料產生預測結果，並計算 accuracy_score。
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print('測試資料筆數:', len(y_test))
print('準確率:', round(accuracy, 3))
print('前 5 筆預測:', [iris.target_names[i] for i in y_pred[:5]])

# Expected:
# 測試資料筆數: 30
# 準確率: 約 0.933 以上
# 前 5 筆預測: 顯示 5 個鳶尾花類別名稱
